# RQ2-v3 end-to-end gradient variance gate — T4×2

One notebook performs the complete diagnostic: (1) reconstruct the frozen Geometry/Resource policies, (2) use the common Uniform seed-3 epoch-100 checkpoint to compute 16 per-batch `(14,14)` interior-gradient Gram matrices on two T4s, and (3) run deterministic four-fold held-out Horvitz–Thompson variance validation on CPU. It does not train a network, take optimizer steps, use accuracy to design a policy, open the test split, or interact with seed-4 training.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
import torch
assert torch.cuda.device_count() == 2, f'Select Kaggle T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

## Resolve frozen RQ2 evidence and common reference checkpoint

In [ ]:
import importlib
import rq2_anchor_placement, rq2_cross_subnet_interaction
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_cross_subnet_interaction = importlib.reload(rq2_cross_subnet_interaction)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
REFERENCE_INPUT = Path('/kaggle/input/notebooks/dyhngg/rq2-seed-3-4-5')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
assert REFERENCE_INPUT.exists(), f'Attach Uniform seed-3 output: {REFERENCE_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-v3-e2e'
)
CHECKPOINT = rq2_cross_subnet_interaction.find_uniform_seed3_checkpoint(
    REFERENCE_INPUT, '/kaggle/working/materialized-uniform-seed3-v3-e2e'
)
CONFIG_PATH = RQ2_ROOT/'resolved_config.yaml'
print('Development root:', RQ2_ROOT)
print('Uniform seed-3 epoch-100 checkpoint:', CHECKPOINT)

## Reconstruct and freeze the exact seed-3 policies

In [ ]:
import rq2_theory_allocation_probe, rq2_probabilistic_support
rq2_theory_allocation_probe = importlib.reload(rq2_theory_allocation_probe)
rq2_probabilistic_support = importlib.reload(rq2_probabilistic_support)
ROOT = Path('/kaggle/working/rq2-v3-cross-batch-e2e')
INTERACTION_DIR = ROOT/'interaction_probe'
ANALYSIS_DIR = ROOT/'v3_cross_batch_variance'
THEORY_ROOT = INTERACTION_DIR/'frozen_policy'/'theory-allocation-probe'
PREVIEW_ROOT = INTERACTION_DIR/'frozen_policy'/'probabilistic-support-preview'
theory = rq2_theory_allocation_probe.run_theory_probe(
    RQ2_ROOT, THEORY_ROOT, waiting_steps=2_000_000
)
preview = rq2_probabilistic_support.build_support_policy(
    RQ2_ROOT, PREVIEW_ROOT, pi_min=1e-8
)
assert theory['training_authorized'] is False and preview['training_authorized'] is False
assert all(value for key, value in theory.items() if key.endswith('_pass'))
assert all(preview['assertions'].values())
GEOMETRY_MARGINALS = THEORY_ROOT/'policy_family_marginals.csv'
RESOURCE_MARGINALS = PREVIEW_ROOT/'support_allocation_marginals.csv'
print('Frozen policy reconstruction complete')

## Stage 1 — compute 16 per-batch Gram matrices on T4×2
GPU 0 handles batches 0–7 and one-step transfer; GPU 1 handles batches 8–15. Both use the same checkpoint, fixed training subset, loss, and BN calibration protocol.

In [ ]:
import scripts.run_cross_subnet_interaction_t4x2 as interaction_runner
interaction_runner = importlib.reload(interaction_runner)
observed_candidates = sorted(Path('/kaggle/input').rglob('dynamic_pilot_width_comparison.csv'))
OBSERVED = observed_candidates[0] if len(observed_candidates) == 1 else None
started = time.perf_counter()
interaction = interaction_runner.run_t4x2_probe(
    checkpoint=CHECKPOINT, config_path=CONFIG_PATH,
    geometry_marginals_path=GEOMETRY_MARGINALS,
    resource_marginals_path=RESOURCE_MARGINALS,
    output_dir=INTERACTION_DIR, dataset_root='/kaggle/working/data',
    observed_comparison_path=OBSERVED, gpu_ids=[0,1], num_batches=16,
)
interaction['git_commit'] = GIT_COMMIT
(INTERACTION_DIR/'metadata.json').write_text(json.dumps(interaction, indent=2)+'\n')
assert interaction['per_batch_interior_gram_shape'] == [16,14,14]
print(f"Gradient stage completed in {(time.perf_counter()-started)/60:.1f} minutes")

## Stage 2 — exact CPU four-fold held-out variance
The Gradient-Oracle-Marginal is fitted on 12 batches per fold and evaluated on four unseen batches. Frozen Geometry/Resource policies are never refitted.

In [ ]:
import rq2_cross_batch_variance_4fold
rq2_cross_batch_variance_4fold = importlib.reload(rq2_cross_batch_variance_4fold)
result = rq2_cross_batch_variance_4fold.run_four_fold_cross_batch_validation(
    INTERACTION_DIR, ANALYSIS_DIR, bootstrap_draws=10_000
)
result['git_commit'] = GIT_COMMIT
(ANALYSIS_DIR/'metadata.json').write_text(json.dumps(result, indent=2)+'\n')
print(json.dumps(result, indent=2))

## Inspect the preregistered gate

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(ANALYSIS_DIR/'fold_variance.csv'))
display(pd.read_csv(ANALYSIS_DIR/'summary.csv'))
display(pd.read_csv(ANALYSIS_DIR/'geo_resource_paired_uncertainty.csv'))
display(pd.read_csv(ANALYSIS_DIR/'oracle_marginal_stability.csv'))
display(Image(filename=str(ANALYSIS_DIR/'variance_by_fold.png')))
display(Image(filename=str(ANALYSIS_DIR/'geo_vs_resource_batch_delta.png')))
display(Image(filename=str(ANALYSIS_DIR/'oracle_marginal_stability.png')))

## Validate and export one combined bundle

In [ ]:
interaction_required = [
    'metadata.json','gram_matrices.npy','fold_assignment.csv',
    'gradient_dot_matrix.csv','gradient_cosine_matrix.csv',
    'gradient_norms_by_batch.csv','policy_expected_effect_by_batch.csv',
]
analysis_required = [
    'metadata.json','gram_matrices.npy','fold_assignment.csv',
    'policy_uniform.csv','policy_resource.csv','policy_geometry.csv',
    'oracle_policy_fold0.csv','oracle_policy_fold1.csv',
    'oracle_policy_fold2.csv','oracle_policy_fold3.csv',
    'batch_variance.csv','fold_variance.csv','summary.csv',
    'bootstrap_geo_minus_resource.npy','geo_resource_paired_uncertainty.csv',
    'oracle_marginal_stability.csv','heldout_geometry_moment_bridge.csv',
    'variance_by_fold.png','geo_vs_resource_batch_delta.png','oracle_marginal_stability.png',
]
missing = [str(INTERACTION_DIR/name) for name in interaction_required if not (INTERACTION_DIR/name).is_file()]
missing += [str(ANALYSIS_DIR/name) for name in analysis_required if not (ANALYSIS_DIR/name).is_file()]
assert not missing, f'Missing end-to-end artifacts: {missing}'
interaction_meta = json.loads((INTERACTION_DIR/'metadata.json').read_text())
analysis_meta = json.loads((ANALYSIS_DIR/'metadata.json').read_text())
assert interaction_meta['training_performed'] is False and interaction_meta['test_used'] is False
assert analysis_meta['network_training'] is False and analysis_meta['oracle_fit_only_on_training_folds'] is True
bundle_path = Path('/kaggle/working/rq2-v3-cross-batch-end-to-end.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in ROOT.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(ROOT))
print('Decision:', analysis_meta['decision'])
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path